# DQN — Deep Q-Network implementado desde Cero

**DQN** (Deep Q-Network) es la evolución natural de Q-Learning cuando el espacio de estados es demasiado grande para una tabla. En lugar de guardar los valores Q en una tabla, los **aproxima con una red neuronal profunda**.

Fue publicado por DeepMind en 2015 y fue el primer algoritmo capaz de aprender a jugar juegos de Atari directamente desde píxeles a nivel humano.

En este notebook implementaremos DQN **desde cero en Python puro** (PyTorch + Gymnasium), aplicado al entorno `CartPole-v1`.

---

## ¿Qué es CartPole?

Un carro sobre un riel con un palo articulado encima. El agente debe mover el carro izquierda/derecha para mantener el palo en equilibrio.

```
        |  (palo)
       ===
    [  carro  ]
________________
```

- **Observación:** 4 valores continuos (posición carro, velocidad carro, ángulo palo, velocidad angular palo)
- **Acciones:** 2 discretas (0 = empujar izquierda, 1 = empujar derecha)
- **Recompensa:** +1 por cada paso que el palo sigue en pie
- **Episodio termina:** si el palo supera 15° de inclinación o el carro sale del riel
- **Objetivo:** aguantar 500 pasos consecutivos

---

## ¿Por qué DQN y no SARSA aquí?

CartPole tiene un espacio de estados **continuo** (4 valores float). Una tabla Q no puede representarlo directamente. DQN usa una red neuronal que acepta esos 4 valores como entrada y devuelve un valor Q por cada acción posible.

---
## Paso 0 — Instalación de dependencias

In [ ]:
import sys
!{sys.executable} -m pip install gymnasium torch numpy matplotlib --quiet
print('Dependencias listas.')

---
## Paso 1 — Importaciones

In [ ]:
import numpy as np
import random
import collections
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Dispositivo: GPU si está disponible, si no CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch    : {torch.__version__}')
print(f'Gymnasium  : {gym.__version__}')
print(f'Dispositivo: {device}')

---
## Paso 2 — Crear el entorno

In [ ]:
env = gym.make('CartPole-v1')
env.reset(seed=SEED)

n_observaciones = env.observation_space.shape[0]  # 4 valores continuos
n_acciones      = env.action_space.n              # 2 acciones discretas

print(f'Espacio de observaciones : {env.observation_space}')
print(f'  → {n_observaciones} valores: [pos_carro, vel_carro, ang_palo, vel_ang_palo]')
print()
print(f'Espacio de acciones      : {env.action_space}')
print(f'  → {n_acciones} acciones: [0=izquierda, 1=derecha]')

---
## Paso 3 — La Red Neuronal Q

En DQN, la función Q ya no es una tabla sino una **red neuronal** que:
- Recibe como **entrada** el estado (4 valores en CartPole)
- Devuelve como **salida** un valor Q por cada acción posible (2 valores en CartPole)

```
Estado (4 valores)  →  [Capa oculta 128]  →  [Capa oculta 128]  →  Q por acción (2 valores)
```

DQN mantiene **dos redes idénticas**:

| Red | Nombre | Función |
|-----|--------|---------|
| Red principal | `policy_net` | Se actualiza en cada paso de entrenamiento |
| Red objetivo  | `target_net` | Copia estable de `policy_net`. Se actualiza cada N pasos |

La `target_net` aporta **estabilidad**: si usáramos una sola red para calcular tanto la predicción como el objetivo, ambos cambiarían al mismo tiempo y el entrenamiento divergiría.

In [ ]:
class RedQ(nn.Module):
    """
    Red neuronal que aproxima la función Q.
    Entrada : estado (n_obs valores)
    Salida  : valor Q para cada acción posible (n_acc valores)
    """

    def __init__(self, n_obs, n_acc, n_ocultas=128):
        super(RedQ, self).__init__()

        self.red = nn.Sequential(
            nn.Linear(n_obs, n_ocultas),   # capa de entrada
            nn.ReLU(),                      # activación no lineal
            nn.Linear(n_ocultas, n_ocultas),# capa oculta
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acc)     # capa de salida: un Q por acción
        )

    def forward(self, x):
        """
        Propagación hacia adelante.
        x: tensor con el estado (batch_size, n_obs)
        retorna: tensor Q (batch_size, n_acc)
        """
        return self.red(x)


# Creamos las dos redes con la misma arquitectura
policy_net = RedQ(n_observaciones, n_acciones).to(device)  # se entrena en cada paso
target_net = RedQ(n_observaciones, n_acciones).to(device)  # copia estable

# Copiamos los pesos de policy_net a target_net
# Al inicio son idénticas
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()  # target_net nunca se entrena directamente

print('Arquitectura de la Red Q:')
print(policy_net)
print()

# Contamos los parámetros entrenables
total_params = sum(p.numel() for p in policy_net.parameters())
print(f'Parámetros entrenables: {total_params:,}')

---
## Paso 4 — El Replay Buffer (Memoria de Experiencias)

El **Replay Buffer** es una de las dos innovaciones clave de DQN (junto con la Target Network).

### ¿Para qué sirve?

Sin él, la red se entrenaría con experiencias consecutivas que están altamente correlacionadas (s1→s2→s3...). Esto hace que el aprendizaje sea inestable.

El Replay Buffer:
1. **Almacena** las transiciones `(s, a, r, s', done)` vividas por el agente
2. En cada paso de entrenamiento, **muestrea un mini-batch aleatorio** de esas transiciones
3. La aleatoriedad **rompe las correlaciones** temporales → entrenamiento estable
4. Permite **reutilizar experiencias** pasadas múltiples veces

```
Buffer:  [(s1,a1,r1,s2), (s2,a2,r2,s3), ..., (sN,aN,rN,sN+1)]
                               ↓
         Muestreo aleatorio de 32 transiciones
                               ↓
                     Mini-batch de entrenamiento
```

In [ ]:
# Una transición es la unidad básica de experiencia:
# (estado, accion, recompensa, estado_siguiente, terminado)
Transicion = collections.namedtuple(
    'Transicion',
    ('estado', 'accion', 'recompensa', 'estado_sig', 'terminado')
)


class ReplayBuffer:
    """
    Buffer circular de experiencias.
    Cuando está lleno, las experiencias más antiguas se descartan.
    """

    def __init__(self, capacidad):
        # deque con maxlen actúa como buffer circular automáticamente
        self.buffer = collections.deque(maxlen=capacidad)

    def guardar(self, estado, accion, recompensa, estado_sig, terminado):
        """Añade una nueva transición al buffer."""
        self.buffer.append(
            Transicion(estado, accion, recompensa, estado_sig, terminado)
        )

    def muestrear(self, batch_size):
        """Devuelve un mini-batch aleatorio de transiciones."""
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


# Ejemplo de funcionamiento
buffer_demo = ReplayBuffer(capacidad=5)
for i in range(7):  # añadimos 7 elementos a un buffer de capacidad 5
    buffer_demo.guardar(f's{i}', i % 2, 1.0, f's{i+1}', False)

print(f'Buffer de capacidad 5 tras añadir 7 elementos:')
print(f'  Tamaño actual: {len(buffer_demo)} (los 2 más antiguos fueron descartados)')
print(f'  Contenido: {[t.estado for t in buffer_demo.buffer]}')

---
## Paso 5 — Hiperparámetros

In [ ]:
# ── Replay Buffer ─────────────────────────────────────────
BUFFER_SIZE    = 10_000  # cuántas transiciones almacena el buffer
BATCH_SIZE     = 64      # cuántas transiciones se usan en cada actualización
LEARNING_START = 1_000   # esperamos hasta tener al menos estas transiciones
                         # antes de empezar a entrenar

# ── Red neuronal y aprendizaje ────────────────────────────
ALPHA          = 1e-3    # tasa de aprendizaje del optimizador Adam
GAMMA          = 0.99    # factor de descuento (importancia del futuro)

# ── Target Network ───────────────────────────────────────
TARGET_UPDATE  = 100     # cada cuántos pasos copiamos policy_net → target_net

# ── Exploración ε-greedy ──────────────────────────────────
EPSILON_INI    = 1.0     # exploración inicial (100% aleatorio)
EPSILON_MIN    = 0.01    # exploración mínima
EPSILON_DECAY  = 0.995   # multiplicador de decaimiento por episodio

# ── Entrenamiento ─────────────────────────────────────────
N_EPISODIOS    = 500     # episodios totales

print('Hiperparámetros:')
print(f'  Buffer size    : {BUFFER_SIZE:,}')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Learning start : {LEARNING_START:,}')
print(f'  Alpha (lr)     : {ALPHA}')
print(f'  Gamma          : {GAMMA}')
print(f'  Target update  : cada {TARGET_UPDATE} pasos')
print(f'  Epsilon        : {EPSILON_INI} → {EPSILON_MIN} (decay={EPSILON_DECAY})')
print(f'  Episodios      : {N_EPISODIOS}')

---
## Paso 6 — Inicializar componentes

In [ ]:
# Replay buffer
buffer = ReplayBuffer(BUFFER_SIZE)

# Optimizador Adam para actualizar los pesos de policy_net
# Solo policy_net se entrena; target_net se actualiza manualmente
optimizer = optim.Adam(policy_net.parameters(), lr=ALPHA)

# Función de pérdida: MSE entre Q predicho y Q objetivo
# MSE = Mean Squared Error = media de (predicción - objetivo)²
criterio = nn.MSELoss()

# Epsilon inicial
epsilon = EPSILON_INI

# Contador global de pasos (para saber cuándo actualizar target_net)
pasos_totales = 0

print('Componentes inicializados:')
print(f'  Replay Buffer  : capacidad {BUFFER_SIZE:,}')
print(f'  Optimizador    : Adam (lr={ALPHA})')
print(f'  Pérdida        : MSELoss')
print(f'  Epsilon inicial: {epsilon}')

---
## Paso 7 — Política ε-greedy con la Red Neuronal

La misma idea que en SARSA, pero ahora los valores Q los calcula la red neuronal.

In [ ]:
def elegir_accion(estado, epsilon):
    """
    Política ε-greedy usando la red neuronal para calcular Q(s, *).

    - Con prob. epsilon  → acción aleatoria (exploración)
    - Con prob. 1-epsilon → acción con mayor Q según policy_net (explotación)

    Parámetros:
        estado  : numpy array con la observación del entorno
        epsilon : probabilidad de exploración

    Retorna:
        acción (int)
    """
    if random.random() < epsilon:
        # Exploración: acción aleatoria
        return env.action_space.sample()

    # Explotación: pasamos el estado por la red y elegimos argmax
    # torch.no_grad(): no necesitamos calcular gradientes aquí
    with torch.no_grad():
        # Convertimos el estado numpy a tensor PyTorch
        # unsqueeze(0): añade dimensión de batch → shape (1, 4)
        estado_tensor = torch.FloatTensor(estado).unsqueeze(0).to(device)

        # Obtenemos los valores Q para todas las acciones
        valores_q = policy_net(estado_tensor)   # shape: (1, 2)

        # Elegimos la acción con mayor valor Q
        return valores_q.argmax(dim=1).item()


# Ejemplo
obs_ejemplo, _ = env.reset()
print(f'Estado de ejemplo: {obs_ejemplo}')
print(f'Acción elegida (epsilon=1.0, aleatoria) : {elegir_accion(obs_ejemplo, epsilon=1.0)}')
print(f'Acción elegida (epsilon=0.0, greedy)    : {elegir_accion(obs_ejemplo, epsilon=0.0)}')

---
## Paso 8 — La Función de Entrenamiento (el núcleo de DQN)

Este es el corazón del algoritmo. En cada llamada:

1. **Muestreamos** un mini-batch del Replay Buffer
2. **Calculamos Q predicho**: pasamos los estados por `policy_net`
3. **Calculamos Q objetivo** (target): usando la ecuación de Bellman con `target_net`
4. **Calculamos la pérdida**: MSE entre predicho y objetivo
5. **Actualizamos los pesos** de `policy_net` mediante backpropagation

### Ecuación de Bellman en DQN:

$$\text{target}(s,a) = r + \gamma \cdot \max_{a'} Q_{\text{target}}(s', a') \cdot (1 - \text{done})$$

El factor `(1 - done)` hace que si el episodio terminó (`done=True`), el valor futuro sea 0 (no hay más recompensas).

In [ ]:
def entrenar_paso():
    """
    Ejecuta un paso de entrenamiento de DQN:
    muestrea del buffer, calcula pérdida y actualiza policy_net.

    Retorna la pérdida del paso (float) o None si el buffer está vacío.
    """
    # No entrenamos hasta tener suficientes experiencias
    if len(buffer) < LEARNING_START:
        return None

    # ── 1. Muestrear mini-batch del buffer ────────────────
    batch = buffer.muestrear(BATCH_SIZE)

    # Desempaquetamos las transiciones en arrays separados
    # Cada variable tiene BATCH_SIZE elementos
    estados     = torch.FloatTensor(np.array([t.estado     for t in batch])).to(device)
    acciones    = torch.LongTensor( np.array([t.accion     for t in batch])).unsqueeze(1).to(device)
    recompensas = torch.FloatTensor(np.array([t.recompensa for t in batch])).to(device)
    estados_sig = torch.FloatTensor(np.array([t.estado_sig for t in batch])).to(device)
    terminados  = torch.FloatTensor(np.array([t.terminado  for t in batch])).to(device)

    # ── 2. Calcular Q predicho por policy_net ─────────────
    # policy_net(estados) → shape (BATCH_SIZE, n_acciones)
    # .gather(1, acciones) → selecciona solo el Q de la acción tomada
    # resultado: shape (BATCH_SIZE, 1)
    q_predicho = policy_net(estados).gather(1, acciones)

    # ── 3. Calcular Q objetivo con target_net ─────────────
    with torch.no_grad():  # no necesitamos gradientes para el target
        # max Q(s', a') sobre todas las acciones posibles
        # .max(1)[0] devuelve el valor máximo por fila → shape (BATCH_SIZE,)
        q_max_sig = target_net(estados_sig).max(1)[0]

        # Ecuación de Bellman:
        # target = r + γ * max_a' Q_target(s', a') * (1 - done)
        # (1 - terminados): si done=True → futuro vale 0
        q_objetivo = recompensas + GAMMA * q_max_sig * (1 - terminados)
        q_objetivo = q_objetivo.unsqueeze(1)  # shape (BATCH_SIZE, 1)

    # ── 4. Calcular pérdida ───────────────────────────────
    # MSE entre lo que predice la red y el valor objetivo
    perdida = criterio(q_predicho, q_objetivo)

    # ── 5. Backpropagation y actualización de pesos ───────
    optimizer.zero_grad()   # limpiamos gradientes previos
    perdida.backward()      # calculamos gradientes

    # Gradient clipping: evita explosión de gradientes
    # Limita la norma de los gradientes a 1.0
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=1.0)

    optimizer.step()        # actualizamos los pesos

    return perdida.item()


print('Función de entrenamiento definida correctamente.')
print('Componentes del paso de entrenamiento:')
print('  1. Muestreo del buffer')
print('  2. Q predicho = policy_net(s).gather(acción)')
print('  3. Q objetivo = r + γ * max target_net(s\')')
print('  4. Pérdida    = MSE(Q_predicho, Q_objetivo)')
print('  5. Backprop   + gradient clipping + optimizer.step()')

---
## Paso 9 — Bucle de Entrenamiento Completo

El bucle DQN por episodio:

```
Para cada episodio:
  1. Reiniciar entorno → estado s
  2. Mientras no termine:
       a. Elegir acción a con ε-greedy
       b. Ejecutar a → obtener (r, s', done)
       c. Guardar (s, a, r, s', done) en el Replay Buffer
       d. Llamar a entrenar_paso() → actualiza policy_net
       e. Cada TARGET_UPDATE pasos → copiar policy_net → target_net
       f. s ← s'
  3. Reducir epsilon
```

In [ ]:
# Métricas de entrenamiento
recompensas_episodio = []
perdidas_episodio    = []
epsilons             = []
duraciones           = []  # número de pasos por episodio

print(f'Iniciando entrenamiento DQN — {N_EPISODIOS} episodios')
print(f'(El agente empieza a aprender tras acumular {LEARNING_START} transiciones)')
print('-' * 65)

for episodio in range(N_EPISODIOS):

    # ── 1. Reiniciar el entorno ──────────────────────────
    estado, info = env.reset()
    recompensa_total = 0
    perdidas_ep = []
    terminado = False
    pasos_ep = 0

    while not terminado:

        # ── a. Elegir acción con ε-greedy ────────────────
        accion = elegir_accion(estado, epsilon)

        # ── b. Ejecutar la acción en el entorno ──────────
        estado_sig, recompensa, terminado, truncado, info = env.step(accion)
        recompensa_total += recompensa
        pasos_ep += 1
        pasos_totales += 1

        # Tratamos truncado como terminado para el buffer
        done_buffer = terminado or truncado

        # ── c. Guardar en el Replay Buffer ───────────────
        buffer.guardar(estado, accion, recompensa, estado_sig, float(done_buffer))

        # ── d. Entrenar un paso ──────────────────────────
        perdida = entrenar_paso()
        if perdida is not None:
            perdidas_ep.append(perdida)

        # ── e. Actualizar Target Network ─────────────────
        # Cada TARGET_UPDATE pasos copiamos policy_net → target_net
        if pasos_totales % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())

        # ── f. Avanzar al siguiente estado ───────────────
        estado = estado_sig

        if truncado:
            terminado = True

    # ── 3. Reducir epsilon ────────────────────────────────
    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

    # Registrar métricas del episodio
    recompensas_episodio.append(recompensa_total)
    duraciones.append(pasos_ep)
    epsilons.append(epsilon)
    perdida_media = np.mean(perdidas_ep) if perdidas_ep else 0
    perdidas_episodio.append(perdida_media)

    # Progreso cada 50 episodios
    if (episodio + 1) % 50 == 0:
        media_100 = np.mean(recompensas_episodio[-100:])
        print(f'Ep {episodio+1:4d}/{N_EPISODIOS} | '
              f'Recomp. media (100ep): {media_100:6.1f} | '
              f'ε={epsilon:.3f} | '
              f'Buffer: {len(buffer):6,} | '
              f'Pasos totales: {pasos_totales:7,}')

print('-' * 65)
print('Entrenamiento completado.')

---
## Paso 10 — Visualización del Entrenamiento

In [ ]:
def media_movil(datos, ventana=50):
    return np.convolve(datos, np.ones(ventana) / ventana, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Entrenamiento DQN — CartPole-v1', fontsize=14, fontweight='bold')

# ── Recompensa por episodio ──────────────────────────────
ax = axes[0, 0]
ax.plot(recompensas_episodio, alpha=0.3, color='steelblue', linewidth=0.8)
if len(recompensas_episodio) >= 50:
    ax.plot(range(49, N_EPISODIOS),
            media_movil(recompensas_episodio),
            color='steelblue', linewidth=2, label='Media móvil (50 ep.)')
ax.axhline(y=500, color='red', linestyle='--', alpha=0.7, label='Objetivo (500)')
ax.set_title('Recompensa por episodio')
ax.set_xlabel('Episodio')
ax.set_ylabel('Recompensa total')
ax.legend()
ax.grid(alpha=0.3)

# ── Duración del episodio ────────────────────────────────
ax = axes[0, 1]
ax.plot(duraciones, alpha=0.3, color='seagreen', linewidth=0.8)
if len(duraciones) >= 50:
    ax.plot(range(49, N_EPISODIOS),
            media_movil(duraciones),
            color='seagreen', linewidth=2, label='Media móvil (50 ep.)')
ax.set_title('Duración del episodio (pasos)')
ax.set_xlabel('Episodio')
ax.set_ylabel('Pasos')
ax.legend()
ax.grid(alpha=0.3)

# ── Pérdida de entrenamiento ─────────────────────────────
ax = axes[1, 0]
perdidas_no_cero = [(i, p) for i, p in enumerate(perdidas_episodio) if p > 0]
if perdidas_no_cero:
    idx, vals = zip(*perdidas_no_cero)
    ax.plot(idx, vals, alpha=0.4, color='darkorange', linewidth=0.8)
    if len(vals) >= 50:
        mm = media_movil(list(vals))
        ax.plot(list(idx)[49:], mm,
                color='darkorange', linewidth=2, label='Media móvil (50 ep.)')
ax.set_title('Pérdida de entrenamiento (MSE)')
ax.set_xlabel('Episodio')
ax.set_ylabel('Pérdida media')
ax.legend()
ax.grid(alpha=0.3)

# ── Decaimiento de epsilon ───────────────────────────────
ax = axes[1, 1]
ax.plot(epsilons, color='mediumpurple', linewidth=2)
ax.fill_between(range(N_EPISODIOS), epsilons, alpha=0.2, color='mediumpurple')
ax.set_title('Decaimiento de ε (exploración → explotación)')
ax.set_xlabel('Episodio')
ax.set_ylabel('ε (epsilon)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Paso 11 — Evaluar el Agente Entrenado

Evaluamos con `epsilon=0` (política completamente greedy, sin exploración).

In [ ]:
def evaluar_agente(policy_net, env, n_episodios=20):
    """
    Evalúa el agente entrenado sin exploración (epsilon=0).
    """
    policy_net.eval()  # modo evaluación (desactiva dropout, batchnorm si los hubiera)
    recompensas = []

    for ep in range(n_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        terminado = False

        while not terminado:
            # epsilon=0: siempre la acción con mayor Q
            accion = elegir_accion(estado, epsilon=0.0)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            if truncado:
                terminado = True

        recompensas.append(recompensa_total)
        print(f'  Episodio {ep+1:2d}: {int(recompensa_total):4d} pasos '
              + ('✓ ÉXITO' if recompensa_total >= 500 else ''))

    policy_net.train()  # volvemos a modo entrenamiento
    print()
    print(f'Recompensa media : {np.mean(recompensas):.1f}')
    print(f'Recompensa máxima: {np.max(recompensas):.0f}')
    print(f'Recompensa mínima: {np.min(recompensas):.0f}')
    return recompensas


print('Evaluación del agente entrenado (epsilon=0):')
print('-' * 40)
resultados = evaluar_agente(policy_net, env, n_episodios=10)

---
## Paso 12 — Guardar y Cargar el Modelo

In [ ]:
# Guardar los pesos de la red entrenada
torch.save(policy_net.state_dict(), 'dqn_cartpole.pth')
print('Modelo guardado en: dqn_cartpole.pth')

# Cargar el modelo guardado
modelo_cargado = RedQ(n_observaciones, n_acciones).to(device)
modelo_cargado.load_state_dict(torch.load('dqn_cartpole.pth', map_location=device))
modelo_cargado.eval()
print('Modelo cargado correctamente.')

# Verificar que funciona igual
obs_test, _ = env.reset()
with torch.no_grad():
    estado_t = torch.FloatTensor(obs_test).unsqueeze(0).to(device)
    q_orig   = policy_net(estado_t)
    q_cargado = modelo_cargado(estado_t)

print(f'Q original : {q_orig.cpu().numpy()}')
print(f'Q cargado  : {q_cargado.cpu().numpy()}')
print(f'Valores idénticos: {torch.allclose(q_orig, q_cargado)}')

---
## Paso 13 — Visualización de los Valores Q Aprendidos

Exploramos qué ha aprendido la red: cómo varían los valores Q en función del ángulo del palo.

In [ ]:
policy_net.eval()

# Variamos el ángulo del palo manteniendo el resto de variables a 0
angulos = np.linspace(-0.2, 0.2, 200)  # rango típico de ángulos en CartPole
q_izq, q_der = [], []

with torch.no_grad():
    for ang in angulos:
        # Estado: [posición=0, vel_carro=0, ángulo=ang, vel_angular=0]
        estado = torch.FloatTensor([[0, 0, ang, 0]]).to(device)
        q_vals = policy_net(estado).cpu().numpy()[0]
        q_izq.append(q_vals[0])   # Q(s, izquierda)
        q_der.append(q_vals[1])   # Q(s, derecha)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Valores Q aprendidos en función del ángulo del palo', fontsize=13, fontweight='bold')

# ── Valores Q por acción ─────────────────────────────────
ax = axes[0]
ax.plot(np.degrees(angulos), q_izq, label='Q(s, izquierda)', color='steelblue', linewidth=2)
ax.plot(np.degrees(angulos), q_der, label='Q(s, derecha)',   color='tomato',    linewidth=2)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Palo vertical')
ax.set_xlabel('Ángulo del palo (grados)')
ax.set_ylabel('Valor Q')
ax.set_title('Q(s, a) por acción')
ax.legend()
ax.grid(alpha=0.3)

# ── Acción elegida (política greedy) ─────────────────────
ax = axes[1]
accion_greedy = [1 if d > i else 0 for i, d in zip(q_izq, q_der)]
ax.scatter(np.degrees(angulos), accion_greedy,
           c=['tomato' if a == 1 else 'steelblue' for a in accion_greedy],
           s=8, alpha=0.7)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_yticks([0, 1])
ax.set_yticklabels(['← Izquierda (0)', 'Derecha (1) →'])
ax.set_xlabel('Ángulo del palo (grados)')
ax.set_title('Política greedy: acción elegida según ángulo')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

policy_net.train()
print('Interpretación:')
print('  - Si el palo cae a la derecha (ángulo +) → el agente empuja a la derecha')
print('  - Si el palo cae a la izquierda (ángulo -) → el agente empuja a la izquierda')
print('  ✓ La política tiene sentido intuitivo')

---
## Resumen

| Paso | Qué hicimos |
|------|-------------|
| 1–2  | Entorno CartPole: estados continuos, 2 acciones |
| 3    | Red neuronal Q: dos redes (policy + target) |
| 4    | Replay Buffer: almacena transiciones, muestrea mini-batches |
| 5–6  | Hiperparámetros e inicialización de componentes |
| 7    | Política ε-greedy sobre la red neuronal |
| 8    | Función de entrenamiento: Bellman + backpropagation |
| 9    | Bucle completo de entrenamiento |
| 10   | Visualización: recompensa, pérdida, duración, epsilon |
| 11   | Evaluación sin exploración |
| 12   | Guardar y cargar el modelo |
| 13   | Visualización de los valores Q aprendidos |

### Las dos innovaciones clave de DQN

| Innovación | Problema que resuelve |
|------------|----------------------|
| **Replay Buffer** | Rompe correlaciones temporales entre experiencias consecutivas |
| **Target Network** | Evita que el objetivo cambie al mismo tiempo que la predicción |

### Diferencia con SARSA

| | SARSA | DQN |
|-|-------|-----|
| Almacena Q en | Tabla | Red neuronal |
| Estados | Discretos | Continuos o discretos |
| Tipo de política | On-policy | Off-policy |
| Memoria | No | Sí (Replay Buffer) |
| Estabilidad | Alta (simple) | Necesita Target Network |

### Siguientes pasos
- Probar con `LunarLander-v2` (8 observaciones, 4 acciones)
- Implementar **Double DQN** (reduce sobreestimación de Q)
- Implementar **Dueling DQN** (separa valor de estado y ventaja de acción)
- Usar `stable-baselines3` para entornos más complejos como Atari

In [ ]:
env.close()
print('Entorno cerrado correctamente.')